In [58]:
!pip install ucimlrepo
# !pip install --upgrade jupyter_client

## Importing libraries

In [59]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

## Getting dataset

In [60]:
retail_data = fetch_ucirepo(id=352)

In [61]:
cols_to_keep = ["InvoiceNo","StockCode","Description"]
retail_df = pd.DataFrame(retail_data.data.original)[cols_to_keep]
retail_df

,InvoiceNo,StockCode,Description
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER
1,536365,71053,WHITE METAL LANTERN
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.
...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE


In [62]:
retail_df.value_counts()

InvoiceNo  StockCode  Description                        
555524     22698      PINK REGENCY TEACUP AND SAUCER         20
C544580    S          SAMPLES                                16
555524     22697      GREEN REGENCY TEACUP AND SAUCER        12
572861     22775      PURPLE DRAWERKNOB ACRYLIC EDWARDIAN     8
C553531    S          SAMPLES                                 7
                                                             ..
552803     21754      HOME BUILDING BLOCK WORD                1
           21755      LOVE BUILDING BLOCK WORD                1
           21792      CLASSIC FRENCH STYLE BASKET GREEN       1
           22066      LOVE HEART TRINKET POT                  1
           20728      LUNCH BAG CARS BLUE                     1
Name: count, Length: 529773, dtype: int64

## Removing null values items

In [63]:
retail_df.isna().sum()

,0
InvoiceNo,0
StockCode,0
Description,1454


In [64]:
retail_df = retail_df.dropna(subset=["Description"])
retail_df

,InvoiceNo,StockCode,Description
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER
1,536365,71053,WHITE METAL LANTERN
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.
...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE


## Getting dataset info

In [65]:
retail_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 540455 entries, 0 to 541908
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   InvoiceNo    540455 non-null  object
 1   StockCode    540455 non-null  object
 2   Description  540455 non-null  object
dtypes: object(3)
memory usage: 16.5+ MB


In [66]:
retail_df.describe()

,InvoiceNo,StockCode,Description
count,540455,540455,540455
unique,24446,3958,4223
top,573585,85123A,WHITE HANGING HEART T-LIGHT HOLDER
freq,1114,2313,2369


## Getting unique invoices and items

In [67]:
len(retail_df["InvoiceNo"].unique())

24446

In [68]:
len(retail_df["StockCode"].unique())

3958

## Removing items which haven't been purchased more than 250 times (as thier are many unique items)

In [69]:
item_counts = retail_df["StockCode"].value_counts()
retail_df = retail_df[retail_df["StockCode"].isin(item_counts[item_counts>=250].index)]
retail_df

,InvoiceNo,StockCode,Description
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER
1,536365,71053,WHITE METAL LANTERN
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.
...,...,...,...
541902,581587,22629,SPACEBOY LUNCH BOX
541903,581587,23256,CHILDRENS CUTLERY SPACEBOY
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL


## Data cleaning

In [70]:
retail_df["StockCode"] = retail_df["StockCode"].str.replace(r"[^0-9]", "", regex=True)
retail_df = retail_df[~retail_df["InvoiceNo"].str.startswith("C")]
retail_df = retail_df[retail_df["StockCode"].astype(str).str.isnumeric()]

### Getting unique item code description

In [71]:
item_counts = retail_df['StockCode'].value_counts()
item_counts.describe()

,count
count,623.000000
mean,493.428571
std,306.527970
min,227.000000
25%,302.000000
50%,388.000000
75%,561.500000
max,3888.000000


### Getting unique item name description

In [72]:
item_counts = retail_df['Description'].value_counts()
item_counts.describe()

,count
count,770.000000
mean,399.228571
std,294.309467
min,1.000000
25%,263.000000
50%,353.000000
75%,492.000000
max,2260.000000


### Since item name and code count is different hence investigating further

In [73]:
retail_df.groupby("StockCode")["Description"].nunique().sort_values(ascending=False).head()

,Description
StockCode,
84997,9
20713,8
23084,7
23343,5
21181,5


### putting name of item for each item code which has max frequency

In [74]:
canonical_desc = (
    retail_df.groupby("StockCode")["Description"]
    .agg(lambda x: x.value_counts().idxmax())
)
retail_df["Description"] = retail_df["StockCode"].map(canonical_desc)

In [75]:
retail_df.groupby("StockCode")["Description"].nunique().sort_values(ascending=False).head()

,Description
StockCode,
85199,1
15036,1
15056,1
16161,1
16237,1


In [76]:
item_counts = retail_df['Description'].value_counts().sort_values(ascending=True)
item_counts

,count
Description,
CHILDS BREAKFAST SET SPACEBOY,227
TRIPLE HOOK ANTIQUE IVORY ROSE,234
PHOTO FRAME 3 CLASSIC HANGING,236
PANTRY ROLLING PIN,239
PARISIENNE CURIO CABINET,242
...,...
LUNCH BAG RED RETROSPOT,1595
REGENCY CAKESTAND 3 TIER,2022
PARTY BUNTING,2079


In [77]:
item_counts = retail_df['StockCode'].value_counts().sort_values()
item_counts

,count
StockCode,
22634,227
23146,234
22796,236
22978,239
23112,242
...,...
20725,1595
22423,2022
47566,2079


### Combining item codes if they have common item name

In [78]:
desc_to_codes = retail_df.groupby("Description")["StockCode"].nunique()
shared = desc_to_codes[desc_to_codes > 1]
print("Descriptions shared by multiple StockCodes:")
print(shared)

Descriptions shared by multiple StockCodes:
Series([], Name: StockCode, dtype: int64)


In [79]:
for desc in shared.index:
    codes = retail_df[retail_df["Description"] == desc]["StockCode"].unique()
    print(f"\nDescription: '{desc}' → StockCodes: {codes}")
    subset = retail_df[retail_df["Description"] == "COLOURING PENCILS BROWN TUBE"]
    dominant_code = subset["StockCode"].value_counts().idxmax()
    retail_df.loc[retail_df["Description"] == "COLOURING PENCILS BROWN TUBE", "StockCode"] = dominant_code

print("Unique StockCodes:", retail_df["StockCode"].nunique())
print("Unique Descriptions:", retail_df["Description"].nunique())

Unique StockCodes: 623
Unique Descriptions: 623


In [80]:
retail_df.value_counts()

InvoiceNo  StockCode  Description                        
555524     22698      PINK REGENCY TEACUP AND SAUCER         20
           22697      GREEN REGENCY TEACUP AND SAUCER        12
572861     22775      PURPLE DRAWERKNOB ACRYLIC EDWARDIAN     8
547498     84596      SMALL DOLLY MIX DESIGN ORANGE BOWL      7
574481     23084      RABBIT NIGHT LIGHT                      6
                                                             ..
553393     21843      RED RETROSPOT CAKE STAND                1
           21731      RED TOADSTOOL LED NIGHT LIGHT           1
           21531      RED RETROSPOT SUGAR JAM BOWL            1
           21428      SET3 BOOK BOX GREEN GINGHAM FLOWER      1
           21935      SUKI  SHOULDER BAG                      1
Name: count, Length: 297012, dtype: int64

In [81]:
retail_df

,InvoiceNo,StockCode,Description
0,536365,85123,WHITE HANGING HEART T-LIGHT HOLDER
1,536365,71053,WHITE METAL LANTERN
2,536365,84406,CREAM CUPID HEARTS COAT HANGER
3,536365,84029,KNITTED UNION FLAG HOT WATER BOTTLE
4,536365,84029,KNITTED UNION FLAG HOT WATER BOTTLE
...,...,...,...
541902,581587,22629,SPACEBOY LUNCH BOX
541903,581587,23256,CHILDRENS CUTLERY SPACEBOY
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL


## Converting the dataframe to one-hot format

In [82]:
retail_items_df = (
    pd.get_dummies(retail_df["StockCode"])
    .groupby(retail_df["InvoiceNo"])
    .max()  # max ensures binary 0/1
)
retail_items_df

,15036,15056,16161,16237,20675,20676,20677,20679,20685,20711,...,85014,85048,85049,85053,85066,85099,85123,85150,85152,85199
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536365,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
536366,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536367,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536368,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536369,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581580,False,False,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
581583,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
581585,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## Importing libraries for apriory algo and rule mining

In [83]:
from mlxtend.frequent_patterns import apriori,association_rules
warnings.filterwarnings(
    "ignore",
    message="datetime.datetime.utcnow() is deprecated"
)
warnings.filterwarnings(
    "ignore",
    message="jupyter_client.session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC)."
)


### getting frequent itemsets

In [84]:
frequent_itemsets = apriori(
    retail_items_df,
    min_support=0.01,
    use_colnames=True
)

In [85]:
frequent_itemsets = frequent_itemsets.sort_values("support",ascending=False)
frequent_itemsets

,support,itemsets
618,0.139978,(85099)
619,0.116288,(85123)
261,0.105102,(22423)
557,0.097926,(47566)
17,0.082573,(20725)
...,...,...
1299,0.010025,"(22699, 22411)"
630,0.010025,"(20711, 21929)"
847,0.010025,"(20728, 22197)"
1157,0.010025,"(22192, 22193)"


### getting rules

In [86]:
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.6
)

rules = rules[rules['lift'] > 1]

rules = rules.sort_values("lift", ascending=False)

### getting non trivial rules (removing weak rules)

In [87]:
rules = rules[
    (rules['confidence'] >= 0.6) &
    (rules['lift'] > 1.2) &
    (rules['support'] >= 0.02)
]
rules = rules.sort_values(
    ["lift", "confidence", "support"],
    ascending=False
)

### frequent itemsets

In [88]:
frequent_itemsets

,support,itemsets
618,0.139978,(85099)
619,0.116288,(85123)
261,0.105102,(22423)
557,0.097926,(47566)
17,0.082573,(20725)
...,...,...
1299,0.010025,"(22699, 22411)"
630,0.010025,"(20711, 21929)"
847,0.010025,"(20728, 22197)"
1157,0.010025,"(22192, 22193)"


### rules

In [89]:
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
38,(22578),(22577),0.027067,0.028439,0.020472,0.756335,26.595218,1.0,0.019702,3.987287,0.989173,0.584337,0.749203,0.738093
37,(22577),(22578),0.028439,0.027067,0.020472,0.719852,26.595218,1.0,0.019702,3.472920,0.990570,0.584337,0.712058,0.738093
19,(22698),"(22699, 22697)",0.040416,0.040521,0.028597,0.707572,17.461730,1.0,0.026959,3.281075,0.982438,0.546371,0.695222,0.706650
17,"(22699, 22697)",(22698),0.040521,0.040416,0.028597,0.705729,17.461730,1.0,0.026959,3.260888,0.982546,0.546371,0.693335,0.706650
16,"(22699, 22698)",(22697),0.031604,0.053554,0.028597,0.904841,16.896019,1.0,0.026905,9.945990,0.971519,0.505597,0.899457,0.719416
12,(22697),(22698),0.053554,0.040416,0.033398,0.623645,15.430744,1.0,0.031234,2.549681,0.988111,0.551394,0.607794,0.725008
11,(22698),(22697),0.040416,0.053554,0.033398,0.826371,15.430744,1.0,0.031234,5.450962,0.974583,0.551394,0.816546,0.725008
18,"(22698, 22697)",(22699),0.033398,0.056244,0.028597,0.856240,15.223564,1.0,0.026719,6.564805,0.966595,0.468453,0.847673,0.682341
15,(23300),(23301),0.039994,0.048172,0.028808,0.720317,14.953079,1.0,0.026882,3.403235,0.971998,0.485333,0.706162,0.659173
33,"(22699, 22423)",(22697),0.027700,0.053554,0.021527,0.777143,14.511516,1.0,0.020043,4.246875,0.957615,0.360424,0.764533,0.589557


### rules generated

In [90]:
print(rules.shape[0],"rules mined")

41 rules mined


### printing all rules

In [91]:
code_to_desc = (
    retail_df[['StockCode', 'Description']]
    .drop_duplicates()
    .set_index('StockCode')['Description']
    .to_dict()
)
def decode_items(itemset):
    return [code_to_desc.get(i, i) for i in itemset]

for _, row in rules.iterrows():

    antecedent = ", ".join(decode_items(row['antecedents']))
    consequent = ", ".join(decode_items(row['consequents']))

    print(f"({antecedent.lower()}) -> {consequent.lower()}")

(wooden star christmas scandinavian) -> wooden heart christmas scandinavian
(wooden heart christmas scandinavian) -> wooden star christmas scandinavian
(pink regency teacup and saucer) -> roses regency teacup and saucer , green regency teacup and saucer
(roses regency teacup and saucer , green regency teacup and saucer) -> pink regency teacup and saucer
(roses regency teacup and saucer , pink regency teacup and saucer) -> green regency teacup and saucer
(green regency teacup and saucer) -> pink regency teacup and saucer
(pink regency teacup and saucer) -> green regency teacup and saucer
(pink regency teacup and saucer, green regency teacup and saucer) -> roses regency teacup and saucer 
(gardeners kneeling pad cup of tea ) -> gardeners kneeling pad keep calm 
(roses regency teacup and saucer , regency cakestand 3 tier) -> green regency teacup and saucer
(regency cakestand 3 tier, green regency teacup and saucer) -> roses regency teacup and saucer 
(pink regency teacup and saucer) -> ro